# 🦀 Day 1: HTTP Client with reqwest

---

## What is reqwest?

**reqwest** is an ergonomic HTTP client for Rust. It provides both blocking and async APIs, making it easy to send HTTP requests, handle responses, and work with JSON.

We'll use the **blocking** client in this notebook since EvCxR runs code synchronously. In a real async application, you'd use the async client with Tokio.

In [ ]:
:dep reqwest = { version = "0.12", features = ["blocking", "json"] }
:dep tokio = { version = "1", features = ["full"] }

// Dependencies loaded. reqwest blocking client is ready to use.

## Blocking vs Async Clients

| Client | Use Case |
|--------|----------|
| `reqwest::blocking` | Scripts, CLI tools, EvCxR notebooks |
| `reqwest` (async) | Web servers, high-concurrency apps |

The blocking client blocks the current thread until the request completes. The async client yields to the runtime while waiting.

## GET Requests

Use `reqwest::blocking::get(url)` for simple GET requests.

In [ ]:
let resp = reqwest::blocking::get("https://httpbin.org/get").expect("request failed");

// Status code
println!("Status: {}", resp.status());

// Headers
println!("Content-Type: {:?}", resp.headers().get("content-type"));

// Body as text
let body = resp.text().expect("failed to read body");
println!("Body (first 200 chars): {}...", &body[..body.len().min(200)]);

## Response Handling: Status, Headers, Body

Check status with `resp.status()`, `resp.status().is_success()`, or match on `StatusCode`. Read the body with `.text()` or `.json()`.

In [ ]:
let resp = reqwest::blocking::get("https://httpbin.org/json").expect("request failed");

if resp.status().is_success() {
    // Parse JSON into serde_json::Value (dynamic)
    let json: serde_json::Value = resp.json().expect("failed to parse JSON");
    println!("JSON response: {:#?}", json);
} else {
    println!("Request failed: {}", resp.status());
}

## POST Requests

Use a `Client` to build POST requests with a body and custom headers.

In [ ]:
let client = reqwest::blocking::Client::new();

let resp = client
    .post("https://httpbin.org/post")
    .header("Content-Type", "application/json")
    .body(r#"{"name": "Rust", "year": 2024}"#)
    .send()
    .expect("request failed");

println!("Status: {}", resp.status());
let json: serde_json::Value = resp.json().expect("failed to parse JSON");
println!("Posted data: {:#?}", json["json"]);

## Error Handling with reqwest

reqwest returns `Result` types. Use `?` in functions or `.expect()` / `.unwrap_or_else()` for handling.

In [ ]:
fn fetch_url(url: &str) -> Result<String, reqwest::Error> {
    let resp = reqwest::blocking::get(url)?;
    let body = resp.text()?;
    Ok(body)
}

// Simpler: just propagate errors
match fetch_url("https://httpbin.org/get") {
    Ok(body) => println!("Got {} bytes", body.len()),
    Err(e) => println!("Error: {}", e),
}

In [ ]:
// Simpler error handling: use expect for quick scripts
let body = reqwest::blocking::get("https://httpbin.org/get")
    .expect("request failed")
    .text()
    .expect("failed to read body");
println!("Body length: {} bytes", body.len());

## Building a Client with Client::builder()

For reusable clients with timeouts, default headers, or connection pooling:

In [ ]:
let client = reqwest::blocking::Client::builder()
    .timeout(std::time::Duration::from_secs(10))
    .user_agent("learn-rust-from-scratch/1.0")
    .build()
    .expect("failed to build client");

let resp = client.get("https://httpbin.org/headers").send().expect("request failed");
let json: serde_json::Value = resp.json().expect("failed to parse JSON");
println!("Request headers we sent: {:#?}", json["headers"]);

## Example: Fetch from a Public JSON API

httpbin.org provides test endpoints. Let's fetch and inspect a response.

In [ ]:
let resp = reqwest::blocking::get("https://httpbin.org/ip")
    .expect("request failed")
    .json::<serde_json::Value>()
    .expect("failed to parse JSON");

println!("Your IP (according to httpbin): {:?}", resp["origin"]);

## 🏋️ Exercise

Write a function `fetch_json_api(url: &str)` that fetches data from a public JSON API and returns the parsed `serde_json::Value`. Handle errors appropriately.

In [ ]:
// YOUR CODE HERE
// fn fetch_json_api(url: &str) -> Result<serde_json::Value, reqwest::Error> {
//     ...
// }
//
// Example usage:
// let data = fetch_json_api("https://httpbin.org/json").expect("fetch failed");
// println!("{:?}", data);

## 🎯 Key Takeaways

1. **reqwest** is an ergonomic HTTP client for Rust with blocking and async APIs
2. Use `reqwest::blocking::get()` for simple GET requests
3. Response: `.status()`, `.headers()`, `.text()`, `.json()`
4. POST: build with `Client::post().header().body().send()`
5. Error handling: `Result` with `?` or `.expect()`
6. `Client::builder()` for timeouts, user-agent, default headers

---

**Tomorrow:** JSON with serde and serde_json — serialize and deserialize structs! 📦